# Linear TS Adapter — Showcase

This notebook demonstrates the **Linear TS search adapter** in ARC, which generates transition-state (TS) guess geometries by internal-coordinate interpolation and family-specific builders.

Each cell below shows a different reaction family:
1. Defines the reaction from SMILES (or adjacency list where needed)
2. Runs the adapter to produce TS guesses
3. Displays the TS as an XYZ string, a prettified coordinate table, and an **interactive 3D viewer**

> **Tip:** Click and drag to rotate the 3D structures. Scroll to zoom.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import py3Dmol
from IPython.display import display, HTML

from arc.species import ARCSpecies
from arc.reaction import ARCReaction
from arc.job.adapters.ts.linear import interpolate_isomerization, interpolate
from arc.species.converter import xyz_to_str, str_to_xyz
from arc.common import get_single_bond_length


# ── Visualisation helpers ──────────────────────────────────────────────

# CPK colours
_ELEMENT_COLORS = {
    'H': '0xFFFFFF', 'C': '0x909090', 'N': '0x3050F8', 'O': '0xFF0D0D',
    'F': '0x90E050', 'S': '0xFFFF30', 'Cl': '0x1FF01F', 'Br': '0xA62929',
}

def _xyz_to_xyz_block(xyz: dict) -> str:
    """Convert an ARC xyz dict to an XYZ-format string block (with atom count header)."""
    n = len(xyz['symbols'])
    lines = [str(n), '']
    for sym, (x, y, z) in zip(xyz['symbols'], xyz['coords']):
        lines.append(f'{sym:2s} {x:12.6f} {y:12.6f} {z:12.6f}')
    return '\n'.join(lines)


def show_ts(xyz: dict, title: str = 'TS Guess', width: int = 500, height: int = 350):
    """Display an interactive 3D view of a TS guess using py3Dmol."""
    viewer = py3Dmol.view(width=width, height=height)
    viewer.addModel(_xyz_to_xyz_block(xyz), 'xyz')

    # Ball-and-stick style with CPK colours
    viewer.setStyle({'stick': {'radius': 0.12, 'colorscheme': 'Jmol'},
                     'sphere': {'scale': 0.25, 'colorscheme': 'Jmol'}})
    viewer.zoomTo()
    viewer.setBackgroundColor('0xF8F8F8')

    display(HTML(f'<b style="font-size:14px;">{title}</b>'))
    viewer.show()


def show_reaction(rxn, ts_xyzs, max_show=2):
    """Show the reaction label, family, number of guesses, and 3D views."""
    family = rxn.family or '(not recognised by RMG)'
    display(HTML(
        f'<div style="background:#e8f4fd;padding:10px;border-radius:6px;margin:4px 0;">'
        f'<b>Reaction:</b> {rxn.label}<br>'
        f'<b>Family:</b> {family}<br>'
        f'<b>TS guesses:</b> {len(ts_xyzs)}</div>'
    ))
    for i, ts in enumerate(ts_xyzs[:max_show]):
        # XYZ string
        xyz_str = xyz_to_str(ts)
        print(f'\n─── TS guess {i} ───')
        print(xyz_str)

        # Prettified coordinate table
        coords = np.array(ts['coords'], dtype=float)
        display(HTML(
            '<table style="font-family:monospace;font-size:12px;border-collapse:collapse;">'
            '<tr style="background:#ddd;"><th>Atom</th><th>x (Å)</th><th>y (Å)</th><th>z (Å)</th></tr>'
            + ''.join(
                f'<tr><td style="padding:2px 8px;">{ts["symbols"][j]}{j}</td>'
                f'<td style="padding:2px 8px;text-align:right;">{coords[j][0]:9.4f}</td>'
                f'<td style="padding:2px 8px;text-align:right;">{coords[j][1]:9.4f}</td>'
                f'<td style="padding:2px 8px;text-align:right;">{coords[j][2]:9.4f}</td></tr>'
                for j in range(len(ts['symbols'])))
            + '</table>'))

        # 3D viewer
        show_ts(ts, title=f'TS guess {i}')

print('Helpers loaded ✓')

---
## 1. Intra H-migration
**`CC[CH2] ⇌ C[CH]C`** — A hydrogen migrates along the carbon backbone through a cyclic TS.

In [ ]:
r = ARCSpecies(label='R', smiles='CC[CH2]')
p = ARCSpecies(label='P', smiles='C[CH]C')
rxn = ARCReaction(r_species=[r], p_species=[p])
ts_xyzs = interpolate_isomerization(rxn)
show_reaction(rxn, ts_xyzs)

---
## 2. Ketoenol tautomerism
**`C=C(O)C(C)=O ⇌ CC(=O)C(C)=O`** — A proton shifts between oxygen and carbon, converting an enol to a ketone.

In [ ]:
r = ARCSpecies(label='R', smiles='C=C(O)C(C)=O')
p = ARCSpecies(label='P', smiles='CC(=O)C(C)=O')
rxn = ARCReaction(r_species=[r], p_species=[p])
ts_xyzs = interpolate_isomerization(rxn)
show_reaction(rxn, ts_xyzs)

---
## 3. 1,2-Shift of sulfur
**`CS[C]1C=CC=C1 ⇌ [S]C1(C)C=CC=C1`** — A sulfur atom migrates between two adjacent carbons via a 3-membered ring TS.

In [ ]:
r = ARCSpecies(label='R', smiles='CS[C]1C=CC=C1')
p = ARCSpecies(label='P', smiles='[S]C1(C)C=CC=C1')
rxn = ARCReaction(r_species=[r], p_species=[p])
ts_xyzs = interpolate(rxn)
show_reaction(rxn, ts_xyzs)

---
## 4. Intra halogen migration
**`FC(F)(Cl)C(F)F ⇌ FC(F)C(F)(F)Cl`** — A chlorine atom migrates between two carbons in a fluorinated ethane.

In [ ]:
r = ARCSpecies(label='R', smiles='FC(F)(Cl)C(F)F')
p = ARCSpecies(label='P', smiles='FC(F)C(F)(F)Cl')
rxn = ARCReaction(r_species=[r], p_species=[p])
ts_xyzs = interpolate(rxn)
show_reaction(rxn, ts_xyzs)

---
## 5. Intra R-Add Exocyclic (radical ring closure)
**`[CH2]C1CC=CC1=C ⇌ [CH2]C12C=CCC1C2`** — A radical CH₂ group attacks across a ring to form a bicyclic product.

In [ ]:
r = ARCSpecies(label='R', smiles='[CH2]C1CC=CC1=C')
p = ARCSpecies(label='P', smiles='[CH2]C12C=CCC1C2')
rxn = ARCReaction(r_species=[r], p_species=[p])
ts_xyzs = interpolate_isomerization(rxn)
show_reaction(rxn, ts_xyzs)

---
## 6. 1,4-Cyclic biradical scission
**`C1=CC=CC=C1 ⇌ [CH]=CC=C[CH]C`** — A 6-membered ring opens by breaking two bonds to form a diradical chain.

In [ ]:
# Singlet biradical — requires adjacency list to preserve radical info
r = ARCSpecies(label='R', smiles='C1=CC=CC=C1')
p = ARCSpecies(label='P', adjlist="""multiplicity 1
1  C u1 p0 c0 {2,S} {6,S} {7,S}
2  C u0 p0 c0 {1,S} {3,D} {8,S}
3  C u0 p0 c0 {2,D} {4,S} {9,S}
4  C u0 p0 c0 {3,S} {5,D} {10,S}
5  C u1 p0 c0 {4,D} {6,S}
6  C u0 p0 c0 {1,S} {5,S} {11,S} {12,S}
7  H u0 p0 c0 {1,S}
8  H u0 p0 c0 {2,S}
9  H u0 p0 c0 {3,S}
10 H u0 p0 c0 {4,S}
11 H u0 p0 c0 {6,S}
12 H u0 p0 c0 {6,S}
""")
rxn = ARCReaction(r_species=[r], p_species=[p])
ts_xyzs = interpolate_isomerization(rxn)
show_reaction(rxn, ts_xyzs)

---
## 7. Radical addition to C≡S (R_Addition_MultipleBond)
**`[CH3] + [C-]#[S+] ⇌ C[C]=S`** — A methyl radical adds to carbon monosulfide.

In [ ]:
r1 = ARCSpecies(label='CH3', smiles='[CH3]')
r2 = ARCSpecies(label='CS', smiles='[C-]#[S+]')
p = ARCSpecies(label='CCS', smiles='C[C]=S')
rxn = ARCReaction(r_species=[r1, r2], p_species=[p])
ts_xyzs = interpolate(rxn)
show_reaction(rxn, ts_xyzs)

---
## 8. HO₂ elimination from peroxy radical
**`CC(O)O[O] ⇌ CC=O + [O]O`** — An OH group and OO• fragment from a peroxy radical, forming an aldehyde and HO₂.

In [ ]:
r = ARCSpecies(label='R', smiles='CC(O)O[O]')
p1 = ARCSpecies(label='P1', smiles='CC=O')
p2 = ARCSpecies(label='P2', smiles='[O]O')
rxn = ARCReaction(r_species=[r], p_species=[p1, p2])
ts_xyzs = interpolate(rxn)
show_reaction(rxn, ts_xyzs)

---
## 9. Intra substitution CS — cyclisation
**`C=CC[S]C ⇌ SC1(C)[CH]C1`** — An intramolecular SN2-like attack forms a thiirane (3-membered ring with sulfur).

In [ ]:
r = ARCSpecies(label='R', smiles='C=CC[S]C')
p = ARCSpecies(label='P', smiles='SC1(C)[CH]C1')
rxn = ARCReaction(r_species=[r], p_species=[p])
ts_xyzs = interpolate(rxn)
show_reaction(rxn, ts_xyzs)

---
## 10. XY elimination (concerted 6-membered ring)
**`CCC(=O)O ⇌ C=C + H₂ + CO₂`** — A concerted elimination producing ethylene, hydrogen gas, and carbon dioxide through a 6-membered ring TS. Uses the dedicated `build_xy_elimination_ts()` builder.

In [ ]:
r = ARCSpecies(label='R', smiles='CCC(=O)O')
p1 = ARCSpecies(label='P1', smiles='C=C')
p2 = ARCSpecies(label='P2', smiles='[H][H]')
p3 = ARCSpecies(label='P3', smiles='O=C=O')
rxn = ARCReaction(r_species=[r], p_species=[p1, p2, p3])
ts_xyzs = interpolate(rxn)
show_reaction(rxn, ts_xyzs, max_show=1)

---
## Summary

The Linear TS adapter supports a wide range of reaction families through a **strategy pipeline**:

| Strategy | Description |
|---|---|
| **Z-mat interpolation** | Blends reactant and product Z-matrices (Type R and Type P branches) |
| **Ring closure** | Rotates torsions to close forming-bond rings |
| **Direct contraction** | Moves terminal groups toward forming-bond partners |
| **3-centre shift** | Repositions a migrating atom between donor and acceptor |
| **Ring scission** | Folds then stretches breaking bonds in ring-opening reactions |
| **Concerted builder** | Simultaneously stretches/contracts multiple bonds for elimination reactions |
| **XY elimination** | Dedicated 6-membered ring builder with dihedral folding |

Each guess goes through **family-specific post-processing** (H triangulation, umbrella flip, rotor staggering) and **validation** (collision, fragment, drift, motif checks) before being returned.